# 👩‍💻 Entrenamiento de Detección de Parkinson (TFM)

Este es tu **Cuaderno Principal**. Substituye al antiguo `train.py`.  
Aquí es donde cargas los audios, lanzas todos tus modelos de deep learning programados, los entrenas en bucle y sacas las notas finales de qué tal han funcionado (Métricas matemáticas).

## 1. Importaciones e Inclusión de la Arquitectura
Al estar en un cuaderno, todavía podemos traernos las definiciones de redes neuronales que escribimos en la carpeta `src/models/` para mantener las cosas super ordenadas.

In [ ]:
import sys
import os
import yaml
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split

# Añadimos la raíz del proyecto para que Colab/Jupyter sepa encontrar "src/"
sys.path.insert(0, os.path.abspath('.'))

from src.models.wavenet import WaveNet
from src.models.transformer_1d import Transformer1D
from src.models.sincnet import SincNet
from src.training.trainer import Trainer
from src.evaluation.metrics import calculate_metrics, plot_roc_curve

## 2. Configuración de Variables
Leemos tu manual de instrucciones `.yaml` para elegir a qué velocidad aprende (Learning rate) o cuántas veces repite (epochs).

In [ ]:
# Vamos a usar WaveNet para este ejemplo
config_path = 'src/training/config/wavenet_pcgita.yaml'
with open(config_path, 'r') as f:
    config = yaml.safe_load(f)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Dispositivo que calculará todo: {device}") # Ojalá sea 'cuda' en CESVIMA/Colab

epochs = config.get('epochs', 10)
batch_size = config.get('batch_size', 16)
lr = config.get('learning_rate', 1e-3)
patience = config.get('early_stopping_patience', 5)

## 3. Preparación de los Audios
A continuación puedes llamar a las funciones del Preprocesamiento, pero para empezar fuerte y asegurar que la red no da fallo vamos a _simular_ unos ruidos que emulen ser las voces cargadas.

In [ ]:
print("Cargando dataset y simulando audios de entrada: 1 Segundo a 16000 hercios")
num_samples = 100
seq_length = 16000 

# X son ondas de sonido aleatorio | y son etiquetas (Parkinson 1, Sano 0)
X = torch.randn(num_samples, 1, seq_length)
y = torch.randint(0, 2, (num_samples,))

# Partimos los datos: 80% Aprende la red | 20% Se lo guardamos a ciegas como test final
indices = torch.arange(num_samples)
train_idx, val_idx = train_test_split(indices, test_size=0.2, random_state=42, stratify=y)

train_dataset = TensorDataset(X[train_idx], y[train_idx])
val_dataset = TensorDataset(X[val_idx], y[val_idx])

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## 4. Creamos Cerebro (Modelo) e Iniciamos el Bucle 
El bucle repite muchas _Epochs_ (épocas) y cuando ve que empeora, la tecnología _Early Stopping_ lo frena del tirón para no gastas minutos de GPU de CESVIMA a lo tonto.

In [ ]:
model_params = config.get('model_params', {})
model = WaveNet(num_classes=2, **model_params.get('wavenet', {}))
model = model.to(device)

# Esto es el clásico 'Optimizador' Adam. Es quien hace que la red entienda que se coló de clase.
optimizer = torch.optim.Adam(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

trainer = Trainer(model, optimizer, criterion, device)
best_val_loss = float('inf')
patience_counter = 0

print("--- COMIENZA EL APRENDIZAJE --- \n")
for epoch in range(epochs):
    train_loss, train_preds, train_labels = trainer.train_epoch(train_loader)
    val_loss, val_preds, val_probs, val_labels = trainer.evaluate(val_loader)
    
    print(f"Época {epoch+1:02d}/{epochs} completada - Loss Entrenamiento: {train_loss:.4f} | Loss de Test Ciego: {val_loss:.4f}")
    
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print(f"\n¡Frenazo automático! Se paró en la Época {epoch+1} porque la máquina dejó de aprender")
            break

## 5. Curvas y Matemáticas
El rendimiento crudo. Si todo ha acabado, escupe el informe en pantalla.

In [ ]:
print("\n--- RESULTADOS FIABLES ---")
metrics = calculate_metrics(val_labels, val_preds, val_probs)

for k, v in metrics.items():
    print(f"{k.capitalize()}: {v:.4f}")
    
if len(torch.unique(torch.tensor(val_labels))) > 1:
    print("Generando Imagen de Curva ROC...")
    plot_roc_curve(val_labels, val_probs, title=f"Desempeño del Modelo", 
                   save_path=f"grafica_roc_final.png")

# Dibujarla en el colab/jupyter si es gráfico:
from IPython.display import Image, display
if os.path.exists('grafica_roc_final.png'):
    display(Image(filename='grafica_roc_final.png'))